# 02 — Compare B0 and zero-shot inpainting (B1)

First inspect B0 on CPU. The diffusion cell then generates **two validation cases,
20 steps each** on an allocated project GPU kernel. No training occurs here.

B0 shifts CIELAB hue/chroma while retaining L and local residuals; it is not a
lightness-editing baseline. B1 uses the pretrained inpainting model without LoRA.
An optional adapter path also enables the trained-model comparison.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/pilot.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the project repository, then reopen this notebook.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = str(ROOT / "results/_hf_cache")
config = json.loads((ROOT / "configs/pilot.json").read_text())
SHARED = Path(config["shared_root"])
DATA = SHARED / "data/CelebAMask-HQ"
MODEL = SHARED / "models/sd15-inpainting"
if not DATA.is_dir() or not MODEL.is_dir():
    raise FileNotFoundError("Check shared_root in configs/pilot.json.")
from IPython.display import display
from PIL import Image
from color_editing.data import select_samples, describe_sample, contact_sheet
from color_editing.runs import create_run, save_json, finish_run, publish_run
print(json.dumps(config, indent=2))
# To try clothing, set config["region"] = "cloth" here before selecting samples.

In [ ]:
from color_editing.data import recolor_lab
from color_editing.pipeline import compare_edits, target_color_for
from color_editing.runs import require_gpu

ADAPTER_DIR = None
# Later: ADAPTER_DIR = Path(".../a completed training run/adapter")

## Fixed validation cases and B0 preview

The split seed fixes the validation IDs; the run seed controls generation. The target is changed if it matches the
provisional source label, so each case requests a different color.

In [ ]:
samples = select_samples(
    DATA, region=config["region"], split="val", limit=config["baseline_samples"],
    seed=config["split_seed"], min_mask_fraction=config["min_mask_fraction"],
    dilation=config["mask_dilation"],
)
print("Validation image IDs:", [s["image_id"] for s in samples])

for sample in samples:
    target = target_color_for(sample, config["target_color"])
    b0 = recolor_lab(sample["image"], sample["mask"], target)
    display(contact_sheet(
        [sample["image"], sample["edit_mask"], b0],
        [f"ID {sample['image_id']} / source: {describe_sample(sample)['source_color']}",
         "inpainting mask", f"B0 -> {target}"],
    ))

## Generate B1 on the GPU and save results

Connect this notebook to Jupyter running inside `-A cis260330p`. The guard checks
the actual kernel's allocation and CUDA visibility. The following cell starts
inference and prints per-case progress plus the sampler progress bar.

B1 and LoRA, when supplied, use identical prompts, masks, seeds, and sampler
settings. We save raw outputs and composites separately. Compositing uses the
original mask to copy the background back; its unchanged background is a
postprocessing result, not evidence of model preservation.

In [ ]:
device = require_gpu()
run_config = {**config, "adapter_directory": str(ADAPTER_DIR) if ADAPTER_DIR else None}
RUN = create_run("baseline-compare", run_config)
print("Saving to:", RUN)
boards = compare_edits(samples, MODEL, RUN, config, adapter_dir=ADAPTER_DIR)
print("Completed:", RUN)

In [ ]:
for path in boards:
    with Image.open(path) as board:
        display(board.copy())
records = [json.loads(line) for line in (RUN / "images.jsonl").read_text().splitlines()]
for record in records:
    print(record["image_id"], record["method"], record.get("variant", ""),
          "outside-mask MAE [0,1]:", record["outside_original_mask_mae_01"],
          "postprocessed:", record["postprocessed"])

## Optional team copy

Share a completed run only after inspecting it. Outputs are not uploaded to
GitHub. This copies within Ocean and refuses to replace an existing run.

In [ ]:
# Optional: uncomment only when this completed run is ready for the team.
# shared_run = publish_run(RUN, SHARED)
# print(shared_run)